In [1]:
!git clone https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git

fatal: destination path 'jacobian-lens-gpt2' already exists and is not an empty directory.


In [3]:
import sys
sys.path.append("/content/jacobian-lens-gpt2")

# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [4]:
import jlens

jlens.configure_logging()

MODEL_NAME = "openai-community/gpt2"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

# LENS_REPO = "neuronpedia/jacobian-lens"
# LENS_REVISION = "qwen-n1000"
# LENS_FILE = {
#     "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
#     # "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
# }[MODEL_NAME]

## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [5]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

HFLensModel(GPT2LMHeadModel, n_layers=12, d_model=768)

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [6]:
# lens = jlens.JacobianLens.from_pretrained(
#     LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
# )
# lens

In [6]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="ckpt.pt"
)
lens.save("jacobian_lens.pt")

[    26s + 26.22s] fit: n_layers=12 d_model=768, fitting 11 source layers (target=L11) on 100 prompts
[    26s +  0.02s]   resuming from checkpoint: 100/100 prompts processed
[    26s +  0.05s] fit: done, 100 prompts


## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [50]:
prompt = "(2 + 3) * 4 = "
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-1])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-1], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  3 logit-lens: ['________________________________________________________________', '------------------------', 'xt', '________', '---------------']
L  3 J-lens:     [' ', '�', '�', 'ドラ', '�']
L  6 logit-lens: ['xt', '________', '________________________________________________________________', '�', 'iced']
L  6 J-lens:     [' ', '�', '�', 'ドラ', '�']
L  9 logit-lens: ['\xa0', '________', '********', '------------------------', 'xt']
L  9 J-lens:     ['�', '�', '�', 'ウス', '�']
L 10 logit-lens: ['\xa0', '________', '�', '【', '_____']
L 10 J-lens:     ['�', '�', '�', '��', 'ア']
model:           ['\xa0', '~~', '【', '????', '�']


## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [8]:
import os, sys
sys.path.append("jacobian-lens-gpt2")
os.chdir("jacobian-lens-gpt2")
os.listdir()

['.git',
 'data',
 'README.md',
 'walkthrough.ipynb',
 'jlens',
 'pyproject.toml',
 'gpt2',
 'tests',
 '.gitignore',
 'experiment.ipynb',
 'assets',
 'LICENSE',
 'uv.lock']

In [49]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = {
    int(k): v for k, v in json.load(gzip.open("assets/qwen_gloss.json.gz")).items()
}

# example = next(e for e in EXAMPLES if e.slug == "multihop")
# prompt = resolve_prompt(example, tokenizer)
prompt = "(2 + 3) * 4 = "

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=1,
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title="Meow",
    description="meow",
    alt_token=None,
)
notebook_iframe(page)

## 5. Render a slice page (served)

For longer prompts prefer `mode="fetch"`: `build_page` writes the data as sidecar files to `out_dir` and the page fetches rank files lazily on pin, so it stays small regardless of how many tokens are tracked.

In [27]:
import os
import threading
from functools import partial
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

from google.colab import output

example = next(e for e in EXAMPLES if e.slug == "ascii-face")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(model, lens, prompt, mask_display=True)
out_dir = Path("slices") / example.slug
out_dir.mkdir(parents=True, exist_ok=True)

page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

# Start one HTTP server for the notebook runtime.
if "_jlens_httpd" not in globals():
    _handler = partial(
        SimpleHTTPRequestHandler,
        directory=os.path.abspath("slices"),
    )
    _jlens_httpd = ThreadingHTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()

port = _jlens_httpd.server_address[1]
print(f"Serving /{example.slug}/ on Colab runtime port {port}")

# Colab proxies the runtime port into your browser. The path is preserved.
output.serve_kernel_port_as_iframe(port, path=f"/{example.slug}/", width="100%", height="800")

Serving /ascii-face/ on Colab runtime port 33625


<IPython.core.display.Javascript object>

In [ ]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

example = next(e for e in EXAMPLES if e.slug == "ascii-face")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(model, lens, prompt, mask_display=True)
out_dir = Path("slices") / example.slug
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/")

### More to explore

A few more prompts are bundled in `jlens.examples.EXAMPLES` — change the `slug` above and see what surfaces, or try a prompt of your own.

In [ ]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")

## 6. Fitting

`fit(model, prompts)` computes `J_l` over the supplied prompts. 100 prompts is enough for a usable lens; the released lenses use 1000. `dim_batch` is the memory knob — each prompt does `ceil(d_model / dim_batch)` backward passes on a retained graph.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="ckpt.pt"
)
lens.save("jacobian_lens.pt")

# 7. Sampling

In [29]:
import torch

def sample(
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    seed=None,
):
    if seed is not None:
        torch.manual_seed(seed)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(hf_model.device)

    with torch.inference_mode():
        output = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,

            # sampling
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,

            # avoid warnings if tokenizer has no explicit pad token
            pad_token_id=tokenizer.eos_token_id,
        )

    # Only decode newly generated tokens
    generated = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

In [42]:
print(sample("The color of the planet fourth from the Sun is "))

 and the way its Sun is dashed in the sky.  The planet's red planet   is  bright red, and the sun's green is dark blue.  And the sun is  a dark blue planet in the sky.  And the sun is  a dark blue planet in the sky.  And the sun is  a dark blue 
